## Erste Schritte vor dem Start, unbedingt ausführen
Im Terminalfenster ab Schritt 2: 
1. Python installieren, Version: ">=3.10", im Installer das Häkchen bei **"Add python.exe to PATH"** setzen
2. Ordner betreten: cd Projektordnername (z.B. Chronos-2)
3. Virtuelle Umgebung erstellen: `python -m venv .venv` (.venv ist der Bezeichner der virtuellen Umgebung, kann auch anders genannt werden)
4. Umgebung aktivieren in Windows: `.\.venv\Scripts\activate`, Mac/Linux: source `.venv/bin/activate`
5. Jupyter in der Umgebung installieren:
`pip install jupyter notebook`
6. Kernel oben rechts aussuchen: Die installierte Python-Version der '.venv' auswählen.
7. Abhängigkeiten installieren : In einer freien Notebook-Zelle vor dem eigentlichen Code `%pip install -r requirements.txt` ausführen.

In [ ]:
%pip install -r requirements.txt

In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

stock_list=["AAPL", "MSFT", "SAP.DE","NVDA","TSLA",
            "SIE.DE","ALV.DE","JPM","GS", "BAYN.DE"]

data=yf.download(stock_list,start="2025-11-01" ,interval="1d")

# data['Close'].head(10)
# data.info()
# data['Close'].isna().sum()

#data['Close']=np.log(data['Close']/data['Close'].shift(1))
data=data.dropna()

print(data["Close"].head())

# data["Close"]=data["Close"].ffill()
# print(data["Close"])

In [ ]:
for stock in stock_list:
    plt.figure(figsize=(12,5))
    plt.plot(data["Close"][stock],label=stock)
    plt.title("Aktienkursverlauf")
    plt.xlabel("Datum")
    plt.ylabel("Preis")
    plt.grid(True)
    plt.legend()
    plt.show()
    

## Forecasting with Chronos-2



In [ ]:
import os
# Use only 1 GPU if available
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from chronos import BaseChronosPipeline, Chronos2Pipeline

# Load the Chronos-2 pipeline
# GPU recommended for faster inference, but CPU is also supported using device_map="cpu"
pipeline: Chronos2Pipeline = BaseChronosPipeline.from_pretrained("amazon/chronos-2", device_map="cpu")

In [ ]:

## Transforming data into the shape of the prediction model

data=yf.Ticker("AAPL").history(start="2025-11-01" ,interval="1d")


#data['Close']=np.log(data['Close']/data['Close'].shift(1))

data=data.dropna()

data_transformed=(data[["Close"]].reset_index().rename(columns={"Date": "timestamp"}))

data_transformed["timestamp"]=(pd.to_datetime(data_transformed["timestamp"]).dt.tz_localize(None))

data_transformed["id"]="AAPL"


data_transformed=(
    data_transformed[["id","timestamp","Close"]]
    .dropna(subset=["timestamp","Close"])
    .sort_values(["id","timestamp"])
    .reset_index(drop=True)
)


print("Input data:",data_transformed.shape)
print(data_transformed.head(10))

In [ ]:
# Prediciton of 15 days of AAPL


pred_df = pipeline.predict_df(data_transformed, prediction_length=15, quantile_levels=[0.1, 0.5, 0.9],timestamp_column='timestamp', id_column='id', target='Close',freq="B")

print("Output dataframe shape:", pred_df.shape)
display(pred_df.head(15))


In [ ]:
# Visualization helper function
def plot_forecast(
    context_df: pd.DataFrame,
    pred_df: pd.DataFrame,
    test_df: pd.DataFrame,
    target_column: str,
    timeseries_id: str,
    id_column: str = "id",
    timestamp_column: str = "timestamp",
    history_length: int = 256,
    title_suffix: str = "",
):
    ts_context = context_df.query(f"{id_column} == @timeseries_id").set_index(timestamp_column)[target_column]
    ts_pred = pred_df.query(f"{id_column} == @timeseries_id and target_name == @target_column").set_index(
        timestamp_column
    )[["0.1", "predictions", "0.9"]]
    ts_ground_truth = test_df.query(f"{id_column} == @timeseries_id").set_index(timestamp_column)[target_column]

    last_date = ts_context.index.max()
    start_idx = max(0, len(ts_context) - history_length)
    plot_cutoff = ts_context.index[start_idx]
    ts_context = ts_context[ts_context.index >= plot_cutoff]
    ts_pred = ts_pred[ts_pred.index >= plot_cutoff]
    ts_ground_truth = ts_ground_truth[ts_ground_truth.index >= plot_cutoff]

    fig = plt.figure(figsize=(12, 3))
    ax = fig.gca()
    ts_context.plot(ax=ax, label=f"historical {target_column}", color="xkcd:azure")
    ts_ground_truth.plot(ax=ax, label=f"future {target_column} (ground truth)", color="xkcd:grass green")
    ts_pred["predictions"].plot(ax=ax, label="forecast", color="xkcd:violet")
    ax.fill_between(
        ts_pred.index,
        ts_pred["0.1"],
        ts_pred["0.9"],
        alpha=0.7,
        label="prediction interval",
        color="xkcd:light lavender",
    )
    ax.axvline(x=last_date, color="black", linestyle="--", alpha=0.5)
    ax.legend(loc="upper left")
    ax.set_title(f"{target_column} forecast for {timeseries_id} {title_suffix}")
    fig.show()

In [ ]:
context_day="2026-10-01"

context_data=data_transformed[data_transformed["timestamp"]<=context_day].copy()

test_data=data_transformed[data_transformed["timestamp"]>context_day].copy()

pred_df = pipeline.predict_df(context_data, prediction_length=3, quantile_levels=[0.1, 0.5, 0.9],timestamp_column='timestamp', id_column='id', target='Close',freq="B")

plot_forecast(
    context_df=context_data,
    pred_df=pred_df,
    test_df=test_data,
    target_column="Close",
    timeseries_id="AAPL",  
    id_column="id",
)



